# Instructions

Make a copy of this colab. Answer all questions by writing code. Run all cells and save output. Download .ipynb and submit it in canvas.

Please check back often for any updates.


2025/11/24 11pm initial version.


# Imports

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '1.0'

# When running with hosted runtime
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')


In [ ]:
import numpy as np
import tensorflow as tf
import keras_hub
import tensorflow_text as text
from google import genai

keras = tf.keras
layers = keras.layers

print('np.__version__:', np.__version__)
print('tf.__version__:', tf.__version__)

print('physical GPUs:', tf.config.list_physical_devices('GPU'))
print('visible GPUs:', tf.config.get_visible_devices('GPU'))


np.__version__: 2.0.2
tf.__version__: 2.19.0
physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
for gpu in tf.config.list_physical_devices('GPU'):
  tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
tf.config.optimizer.set_jit(False)


# Q1 (9 points) BERT Sentiment Analysis


Q1(a) (1 point) What does BERT acronym mean? What kind of a model is it?

Q1(b) (2 points) How is it trained -- describe the two tasks for which it is trained. Is this supervised learning?

Q1(c) (3 points) The goal is to build an IMDB review classifier (binary classifier). Download IMDB movie review dataset, split into train/validation/test. Get BertTokenizer and BertBackbone model from preset (pretrained 'bert_base_en' 100M) from keras_hub. Write your own Preprocessor. Freeze the backbone. Add a few layers of simple MLP on top of the backbone that use all the outputs of the backbone, so that you have ~50k trainable params. (Don't use BertTextClassifierPreprocessor / BertTextClassifier directly.)

Q1(d) (1 point) How many trainable and nontrainable params do you have in your model?

Q1(e) (2 point) Train the model for 5 epochs. How long did the training take? Report precision/recall/accuracy on test set.



Your Answers:

Q1(a) (1 point) = ...


Q1(b) (2 points) = ...

In [ ]:
Q1(a) (1 point) What does BERT acronym mean? What kind of a model is it?
Q1(a) (1 point) =

Q1(a) (1 point) What does BERT acronym mean? What kind of a model is it?
BERT - Bidirectional Encoder Representations from Transformers
   1) reads text from right-to-left and left-to-right.
   2) deep learning model based on the Transformer architecture.
   3) uses only encoder component.
   4) each word has a numeric representation, which change based on neighbouring words.  


 Q1(b) How is it trained -- describe the two tasks for which it is trained. Is this supervised learning?
  
    1. Involves 2 phases of training.
    2. Pre-training Phase- Unsupervised (Creates understanding of language)
       Training tasks -
        a. Masked Language Model - Mask some words in a sentence and then predict the missing words.
        b. Next Sentence Prediction (NSP) / Causal Language Modeling (CLM)
          a. NSP - reads two sentences, decides if sentence follows the first.
          b. CLM - predicts next words in a sentence, learns to complete sentences or packages.
    3. Fine Tuning Phase-
      a. adapted to specific downstream tasks. eg. sentiment analysis
      b. involves supervised learning, where labeled datasets are used.
    4. It uses a hybrid learning approach that combines of both supervised and unsupervised learning, depending on the phase of its usage.

## Get IMDB dataset

Q1(a) (1 point) = Bert - Bidirectional Encoder

In [ ]:
import shutil
import os
import tensorflow as tf

def fetch_dataset():
  cache_dir = 'datasets'
  dataset = f'{cache_dir}/aclImdb_v1_extracted'

  if os.path.exists(dataset):
    return dataset

  url = 'https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
  print(f'fetching {url}')

  dataset = tf.keras.utils.get_file('aclImdb_v1.tar.gz', url,
                                    untar=True, cache_dir=cache_dir,
                                    cache_subdir='')
  # remove unused folders to make it easier to load the data
  remove_dir = os.path.join(f'{dataset}/aclImdb/train', 'unsup')
  shutil.rmtree(remove_dir)
  return dataset
dataset = fetch_dataset()
print(dataset)


fetching https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 16s 0us/step
/tmp/.keras/aclImdb_v1_extracted


In [ ]:
print(dataset)

/tmp/.keras/aclImdb_v1_extracted


In [ ]:
# Q1(c) (1st point)
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 1237

raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    f"{dataset}/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed
)

class_names = raw_train_ds.class_names
print("Class names:", class_names)

train_ds = raw_train_ds.cache().prefetch(AUTOTUNE)

val_ds = tf.keras.utils.text_dataset_from_directory(
    f"{dataset}/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed
).cache().prefetch(AUTOTUNE)

test_ds = tf.keras.utils.text_dataset_from_directory(
    f"{dataset}/aclImdb/test",
    batch_size=batch_size
).cache().prefetch(AUTOTUNE)


Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Class names: ['neg', 'pos']
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.
Found 25000 files belonging to 2 classes.


In [ ]:
for x, y in train_ds.take(1):
    print(x.dtype, x.shape)
    print(y.dtype, y.shape)

<dtype: 'string'> (32,)
<dtype: 'int32'> (32,)


In [ ]:
def print_train_sample():
  for text_batch, label_batch in train_ds.take(1):
    for i in range(3):
      print(f'Review: {text_batch.numpy()[i]}')
      label = label_batch.numpy()[i]
      print(f'Label : {label} ({class_names[label]})')

print_train_sample()

Review: b"I was unlucky enough to have seen this at the Sidewalk Film Festival. Sidewalk as a whole was a disappointment and this movie was the final nail in the coffin. Being a devout fan of Lewis Carroll's 'Alice' books I was very excited about this movie's premier, which only made it that much more uncomfortable to watch. Normally I'm enthusiastic about modern re-tellings if they are treated well. Usually it's interesting to see the parallels between the past and present within a familiar story. Unfortunately this movie was less of a modern retelling and more of a pop culture perversion. The adaptation of the original's characters seemed juvenile and usually proved to be horribly annoying. It probably didn't help that the actors weren't very good either. Most performances were ridiculously over the top, which I assume was either due to bad direction or an effort to make up for a bad script. I did not laugh once through out the duration of the film. All of the jokes were outdated ref

## Get BERT 100M parameters model

Get the pretrained version from keras_hub.models...from_preset(...)

In [ ]:
model_name = 'bert_base_en'

In [ ]:
import keras_hub
tokenizer = keras_hub.tokenizers.BertTokenizer.from_preset("bert_base_en")
backbone = keras_hub.models.BertBackbone.from_preset("bert_base_en")
backbone.trainable=False

100%|██████████| 762/762 [00:00<00:00, 1.41MB/s]


100%|██████████| 208k/208k [00:00<00:00, 272kB/s]


100%|██████████| 457/457 [00:00<00:00, 927kB/s]


100%|██████████| 414M/414M [00:28<00:00, 15.5MB/s]


In [ ]:
text_test = tf.constant(['this is such an amazing movie!'])

text_preprocessed = tokenizer(text_test)
text_preprocessed


[[1142, 1110, 1216, 1126, 6929, 2523, 106]]

In [ ]:
backbone.summary()


Model: "bert_backbone"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_embedding     │ (None, None, 768) │ 22,268,928 │ token_ids[0][0]   │
│ (ReversibleEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ segment_ids         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding  │ (None, None, 768) │    393,216 │ token_embedding[… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ segment_embedding   │ (None, None, 768) │      1,536 │ segment_ids[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_add      │ (None, None, 768) │          0 │ token_embedding[… │
│ (Add)               │                   │            │ position_embeddi… │
│                     │                   │            │ segment_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_layer_n… │ (None, None, 768) │      1,536 │ embeddings_add[0… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embeddings_dropout  │ (None, None, 768) │          0 │ embeddings_layer… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ padding_mask        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_0 │ (None, None, 768) │  7,087,872 │ embeddings_dropo… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_1 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_2 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_3 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_4 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_5 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_layer_6 │ (None, None, 768) │  7,087,872 │ transformer_laye… │
│ (TransformerEncode… │                   │            │ padding_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 108,310,272 (413.17 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 108,310,272 (413.17 MB)

In [ ]:
# Let's check backbone.input to see how to write preprocessor
backbone.input

{'token_ids': <KerasTensor shape=(None, None), dtype=int32, sparse=False, ragged=False, name=token_ids>,
 'segment_ids': <KerasTensor shape=(None, None), dtype=int32, sparse=False, ragged=False, name=segment_ids>,
 'padding_mask': <KerasTensor shape=(None, None), dtype=int32, sparse=False, ragged=False, name=padding_mask>}

In [ ]:
# Q1(c) (2nd point)
from tensorflow.keras import layers
max_seq_len = 128

class BertPreprocessor(layers.Layer):
  def __init__(self):
    super().__init__()
    self.max_seq_len = max_seq_len
    self.tokenizer = tokenizer
    self.packer = keras_hub.layers.StartEndPacker(
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        sequence_length=self.max_seq_len,
    )

  def call(self, inputs):
    token_ids = self.tokenizer(inputs)

    x = self.packer(token_ids)
    segment_ids = tf.zeros_like(x)
    padding_mask = tf.cast(tf.not_equal(x, self.tokenizer.pad_token_id), tf.int32)
    x = {
        'token_ids': x,
        'segment_ids':  segment_ids,
        'padding_mask': padding_mask
    }
    return x

bert_preprocessor = BertPreprocessor()

In [ ]:
inputs = bert_preprocessor(text_test)
inputs

{'token_ids': <tf.Tensor: shape=(1, 128), dtype=int32, numpy=
 array([[ 101, 1142, 1110, 1216, 1126, 6929, 2523,  106,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0]], dtype=int32)>,
 'segment_ids': <tf.Tensor: shape=(1, 128), d

In [ ]:
outputs = backbone(inputs)
outputs

{'sequence_output': <tf.Tensor: shape=(1, 128, 768), dtype=float32, numpy=
 array([[[ 3.47391397e-01,  1.88405454e-01, -2.63694189e-02, ...,
          -9.43800211e-02,  3.92732948e-01,  1.06159449e-01],
         [-3.29975754e-01,  1.07954457e-01,  1.42662287e-01, ...,
           2.01788932e-01,  4.49127913e-01,  5.18322349e-01],
         [ 4.73515838e-01,  6.19435489e-01,  6.82160079e-01, ...,
          -2.88421363e-02,  4.20234859e-01,  9.28305089e-01],
         ...,
         [-1.22584656e-01,  2.82323569e-01,  2.56040424e-01, ...,
          -3.26336116e-01,  2.40193710e-01,  1.37644215e-02],
         [-3.81801277e-02,  1.31512314e-01,  3.24666619e-01, ...,
          -2.74626613e-01,  3.60071123e-01,  1.27429128e-01],
         [ 2.55584717e-04,  1.26830071e-01,  2.69849896e-01, ...,
          -5.76839261e-02,  3.76305580e-01,  2.17616916e-01]]],
       dtype=float32)>,
 'pooled_output': <tf.Tensor: shape=(1, 768), dtype=float32, numpy=
 array([[-7.60098398e-01,  5.21006167e-01,  9.999

In [ ]:
print(outputs.keys())
for key in outputs.keys():
  print(key, outputs[key].shape)

dict_keys(['sequence_output', 'pooled_output'])
sequence_output (1, 128, 768)
pooled_output (1, 768)


In [ ]:
backbone.trainable=False

In [ ]:
# Q1(c) (3rd point)
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

tf.config.optimizer.set_jit(False)
keras.backend.clear_session()

def build_model1():
    inputs = layers.Input(shape=(), dtype=tf.string, name="text")

    x = bert_preprocessor(inputs)
    x = backbone(x)
    # shape = [None, 768]
    z = x['pooled_output']
    x = x['sequence_output']

    seq_pool = layers.GlobalAveragePooling1D()(x)

    combined = layers.Concatenate()([z, seq_pool])

    x = layers.Dropout(0.2)(combined)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    return keras.Model(inputs, outputs)

model1 = build_model1()

model1.compile(
    loss='binary_crossentropy',
    optimizer='adamw',
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
    jit_compile=False
)

#Q1.d
model1.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_preprocessor   │ [(None, 128),     │          0 │ text[0][0]        │
│ (BertPreprocessor)  │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_backbone       │ [(None, 768),     │ 108,310,2… │ bert_preprocesso… │
│ (BertBackbone)      │ (None, 128, 768)] │            │ bert_preprocesso… │
│                     │                   │            │ bert_preprocesso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 768)       │          0 │ bert_backbone[2]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ bert_backbone[2]… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1536)      │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │     49,184 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      1,056 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 108,360,545 (413.36 MB)

 Trainable params: 50,273 (196.38 KB)

 Non-trainable params: 108,310,272 (413.17 MB)

In [ ]:
# Q1(e) (1st point)
import time

start_time = time.time()

history1 = model1.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

end_time = time.time()

time_taken = end_time - start_time

print(f"Time taken: {time_taken:.2f} seconds")
print(f"Time taken: {time_taken/60:.2f} minutes")


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.7875 - loss: 0.4558 - precision: 0.7864 - recall: 0.7828 - val_accuracy: 0.8052 - val_loss: 0.4421 - val_precision: 0.7681 - val_recall: 0.8665
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.7867 - loss: 0.4631 - precision: 0.7880 - recall: 0.7780 - val_accuracy: 0.8096 - val_loss: 0.4301 - val_precision: 0.7959 - val_recall: 0.8256
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.7849 - loss: 0.4623 - precision: 0.7847 - recall: 0.7788 - val_accuracy: 0.8116 - val_loss: 0.4243 - val_precision: 0.8021 - val_recall: 0.8203
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.7907 - loss: 0.4593 - precision: 0.7903 - recall: 0.7851 - val_accuracy: 0.8084 - val_loss: 0.4260 - val_precision: 0.8052 - val_recall: 0.8065
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.7895 - loss: 0.4557 - precision: 0.7928 - recall: 0.7776 - val_accuracy: 0.8140 - val_los

In [ ]:
# Q1(e) (2nd point)

test_eval1 = model1.evaluate(test_ds, verbose=1)

metrics = dict(zip(model1.metrics_names, test_eval1))
metrics



782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 35ms/step - accuracy: 0.8105 - loss: 0.4261 - precision: 0.8042 - recall: 0.8168


{'loss': 0.4265401065349579, 'compile_metrics': 0.8091599941253662}

# Q2 (5 points) LoRA model


Q2(a) (1 point) Explain what LoRA is.

Q2(b) (1 point) Use the same tokenizer/backbone as before. But now enable_lora(2) on the backbone. Add a very small MLP layer.

Q2(c) (1 point) What are the number of trainable and non-trainable params in your new model?

Q2(d) (2 points) Train the model for 3 epocs. How long did it take? Report precision/recall/accuracy on test set.


Your Answers:

Q2(a) (1 point) = ...


In [ ]:

LoRA = Low-Rank Adaptation
1) used to adapt machine learning models to new techniques
1) is a technique of fine-tuning large pre-trained models. eg.: LLMs
2) freezes original weights
3) adds low rank, comparatively smaller size trainable parameters.(rank decomposition matrices)

In [ ]:
for v in backbone.trainable_weights:
    print(v.name)
#LORA NOT ENABLED

In [ ]:
keras.backend.clear_session()
backbone.enable_lora(rank=2)

In [ ]:
# Q2(b) (1 point)
# enable lora
for v in backbone.trainable_weights:
    print(v.name)
#LORA ENABLED

bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b
bias
lora_kernel_a
lora_kernel_b


In [ ]:
# Q2(c) (1st point)
def build_model2():
  inputs = layers.Input(shape=(), dtype=tf.string, name="text")
  x = bert_preprocessor(inputs)
  x = backbone(x)
  z = x['pooled_output']
  outputs = layers.Dense(1, activation='sigmoid')(z)

  return keras.Model(inputs, outputs)

model2 = build_model2()
model2.compile(loss='binary_crossentropy', optimizer='adamw', metrics=[
    keras.metrics.BinaryAccuracy(name="accuracy"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
],
    jit_compile=False
)
model2.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_preprocessor   │ [(None, 128),     │          0 │ text[0][0]        │
│ (BertPreprocessor)  │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_backbone       │ [(None, 768),     │ 108,755,7… │ bert_preprocesso… │
│ (BertBackbone)      │ (None, 128, 768)] │            │ bert_preprocesso… │
│                     │                   │            │ bert_preprocesso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1)         │        769 │ bert_backbone[6]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 108,756,481 (414.87 MB)

 Trainable params: 464,641 (1.77 MB)

 Non-trainable params: 108,291,840 (413.10 MB)

In [ ]:
# Q2(d) (1st point)
import time
start_time = time.time()

history2 = model2.fit(
    train_ds,
    epochs=3,
    validation_data=val_ds
)

end_time = time.time()

time_taken = end_time - start_time

print(f"Time taken: {time_taken:.2f} seconds")
print(f"Time taken: {time_taken/60:.2f} minutes")

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 87s 99ms/step - accuracy: 0.8430 - loss: 0.3598 - precision: 0.8451 - recall: 0.8363 - val_accuracy: 0.8608 - val_loss: 0.3134 - val_precision: 0.8192 - val_recall: 0.9209
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.8754 - loss: 0.2996 - precision: 0.8676 - recall: 0.8837 - val_accuracy: 0.8714 - val_loss: 0.3007 - val_precision: 0.8369 - val_recall: 0.9181
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.8968 - loss: 0.2599 - precision: 0.8897 - recall: 0.9040 - val_accuracy: 0.8764 - val_loss: 0.2953 - val_precision: 0.8620 - val_recall: 0.8921
Time taken: 203.63 seconds
Time taken: 3.39 minutes


In [ ]:
# Training took ?? mins.
Time taken: 203.63 seconds
Time taken: 3.39 minutes
#Used A100 GPU

In [ ]:
# Q2(d) (2nd point)

test_eval2 = model2.evaluate(test_ds, verbose=1)

metrics = dict(zip(model2.metrics_names, test_eval2))
metrics

test_eval2

782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.8699 - loss: 0.3056 - precision: 0.8668 - recall: 0.8754


[0.3008815348148346, 0.8738800287246704, 0.869359016418457, 0.8799999952316284]

# Q3 Gemini Agent (6 points)

Use Gemini API to build a agent that takes movie reviews from the IMDB test set and outputs sentiment value.

Make sure to give some examples to gemini in your prompt from the training data (but not all of training data). Make sure that the output is 0 (negative) or 1 (positive). Measure precision/recall on a very small random subset of test set (10 reviews) (send all 10 reviews in one api call) (you will have to parse the output and measure precision/recall/accuracy on your own, don't rely on gemini to compute these).


In [ ]:
!pip install google-genai tensorflow

In [ ]:
import random
import json
import numpy as np
from google import genai

client = genai.Client(api_key="YOUR_API_KEY")


In [ ]:
# Q3 (1st point)
from tensorflow.keras.datasets import imdb
from datasets import load_dataset

# Get 2 examples from train_ds with label=0 (negative) and 2 with label=1 (positive)
train_ds = load_dataset("imdb")["train"]

examples_neg = []
examples_pos = []

for i in train_ds:
    if i["label"] == 0 and len(examples_neg) < 2:
        examples_neg.append(i)
    elif i["label"] == 1 and len(examples_pos) < 2:
        examples_pos.append(i)
    if len(examples_neg) == 2 and len(examples_pos) == 2:
        break

print(" Negetive examples: Count: ",len(examples_neg)," - ",examples_neg)
print(" Positive examples: Count: ",len(examples_pos)," - ",examples_pos)

 Negetive examples: Count:  2  -  [{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex a

In [ ]:
# Q3 (2nd point)

# Get 10 test examples and labels.

test_ds = load_dataset("imdb")["test"]

test_examples = []
test_labels = []

indices = random.sample(range(len(test_ds)), len(test_ds))
count_0=0
count_1=0

for i in indices:
  if(test_ds[i]['label']==0) and count_0<5:
    test_examples.append(test_ds[i]['text'])
    test_labels.append(test_ds[i]['label'])
    count_0+=1
  if(test_ds[i]['label']==1) and count_1<5:
    test_examples.append(test_ds[i]['text'])
    test_labels.append(test_ds[i]['label'])
    count_1+=1
  if count_0==5 and count_1==5:
    break

print(" test_examples: ",test_examples)
print(" test_labels: ",test_labels)

# Add code

 test_examples:  ['Whoever likened this one to RAIDERS OF THE LOST ARK certainly knew whereof he spoke. He might, as well, have likened it to some of the adventures of the pulp heroes that followed. "Kay Hoog" reminds one more than a little of both Lamont Cranston (The Shadow) and Clark Savage (Doc Savage). (The Shadow, quintessential man of mystery- and the very first "Dark Knight"- was also thought to be one Kent Allard. If one were to take Savage\'s first name first and add to it the Kent, you end up with- voila- Clark Kent. Funny, innit?) Like Indiana Jones, Hoog isn\'t above pilfering the artifacts of an ancient civilization (though his thefts are often more blatant and less "charmingly roguish" than Jones\'s). Unfortunately, this two-parter is a far cry from subsequent serials (from any era) in terms of overall quality. One of the first indications that something is amiss vis a vis the cinematic storytelling is a scene where desperados on horseback, quite literally breathing down

In [ ]:
# Q3 (3rd point)

# Build prompt/query for the model using the examples.

prompt = """
You are a sentiment classifier.
Return ONLY 0 or 1.
0 = negative review
1 = positive review

Below are some example reviews with correct labels.
"""
# Add negative examples
for ex in examples_neg:
    prompt += f'\nReview: "{ex["text"]}"\nSentiment: 0\n'

# Add positive examples
for ex in examples_pos:
    prompt += f'\nReview: "{ex["text"]}"\nSentiment: 1\n'

prompt += """
Now you will be given new reviews.
Classify each one as 0 or 1.
Return ONLY a JSON list like: [0,1,0,1,...]
"""

In [ ]:
prompt

'\nYou are a sentiment classifier.\nReturn ONLY 0 or 1.\n0 = negative review\n1 = positive review\n\nBelow are some example reviews with correct labels.\n\nReview: "I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /

In [ ]:
# Q3 (4th point)
# Build final prompt sent to Gemini

final_prompt = prompt + "\nNow classify the following 10 reviews.\n"
final_prompt += "Return ONLY a JSON list of 10 numbers (0 or 1), in order.\n\n"

for i, review in enumerate(test_examples):
    final_prompt += f"Review {i+1}: {review}\n\n"

final_prompt += "Output:"

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=final_prompt
)
print(final_prompt)


You are a sentiment classifier.
Return ONLY 0 or 1.
0 = negative review
1 = positive review

Below are some example reviews with correct labels.

Review: "I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />Wh

In [ ]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='[0, 1, 1, 1, 1, 1, 0, 0, 0, 0]'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='m4U8abvYNZ7nxN8P9JO8qAU',
  sdk_http_response=HttpResponse(
    headers=<dict len=10>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=30,
    prompt_token_count=3847,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=3847
      ),
    ],
    thoughts_token_count=2790,
    total_token_count=6667
  )
)

In [ ]:
# Q3 (5th point)

# parse the response to get the predicted labels
pred_labels = response.text
print("Model raw output:", pred_labels)

Model raw output: [0, 1, 1, 1, 1, 1, 0, 0, 0, 0]


In [ ]:
# Q3 (6th point)

# compare predicted to test_labels, compute precision / recall / accuracy.

import numpy as np

# Convert to numpy for easy vector operations
test_labels_np = np.array(test_labels)
pred_labels_np = np.array(json.loads(pred_labels))

print(test_labels_np)
print(pred_labels_np)


positive_count=0
for i in range(0,len(test_labels_np)):
    if(test_labels_np[i]==pred_labels_np[i]):
        positive_count+=1
accuracy = positive_count/len(test_labels_np)
print("Accuracy = ",accuracy)

true_positive = ((pred_labels_np == 1) & (test_labels_np == 1)).sum()
true_negetive = ((pred_labels_np == 0) & (test_labels_np == 0)).sum()
false_positive = ((pred_labels_np == 1) & (test_labels_np == 0)).sum()
false_negetive = ((pred_labels_np == 0) & (test_labels_np == 1)).sum()

precision = true_positive / (true_positive + false_positive)
print("Precision:", precision)

recall = true_positive / (true_positive + false_negetive )
print("Recall:", recall)


[0 1 1 1 1 1 0 0 0 0]
[0 1 1 1 1 1 0 0 0 0]
Accuracy =  1.0
Precision: 1.0
Recall: 1.0
